In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[0], documents[-1]

({'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'ab183bd688',
  'course': 'machine-learning-zoomcamp',
  'section': 'Miscellaneous',
  'question': "My homework answer doesn't match any of the options",
  'answer': "Common causes, in order of frequency:\n\n1. Wrong column slice or filter — apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform ap

In [3]:
documents_llm = []
course_name = 'llm-zoomcamp'
for doc in documents:
    if doc['course'] == course_name:
        documents_llm.append(doc)     
print(f"total FAQ records for course:{course_name} = {len(documents_llm)}")

total FAQ records for course:llm-zoomcamp = 103


In [4]:
documents = documents_llm

In [5]:
doc = documents[0]

In [6]:
doc['id'], doc['question'], doc['answer']

('74eb249bbf',
 'I just discovered the course. Can I still join?',
 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.')

In [7]:
## generate structured response using pydantic
from pydantic import BaseModel
class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [11]:
response = {
    'questions':['what is your name?', 'another question?']
}

In [12]:
r = Questions.model_validate(response)

In [13]:
type(r)

__main__.Questions

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [14]:
import json

user_prompt = json.dumps(doc)

In [17]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [18]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [19]:
print(messages)

[{'role': 'developer', 'content': "You emulate a student who's taking our course.\nFormulate 5 questions this student might ask based on a FAQ record. The record\nshould contain the answer to the questions, and the questions should be complete and not too short.\nIf possible, use as fewer words as possible from the record.\n\nThe output should resemble how people ask questions\non the internet. Not too formal, not too short, not too long."}, {'role': 'user', 'content': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'}]


In [20]:
## use response.parse to elicit response in our pydantic structure
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [21]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I found out about it late?', 'If I join now, am I still eligible for a certificate?', 'What do I need to do to get a certificate if I’m starting late?', 'Is it okay to start the course after it already began?', 'Can late joiners still submit the project for certification?']


In [22]:
print(result.questions)

['Can I still join the course if I found out about it late?', 'If I join now, am I still eligible for a certificate?', 'What do I need to do to get a certificate if I’m starting late?', 'Is it okay to start the course after it already began?', 'Can late joiners still submit the project for certification?']


In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found out about it late?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to get a certificate if I’m starting late?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course after it already began?',
  'document': '74eb249bbf'},
 {'question': 'Can late joiners still submit the project for certification?',
  'document': '74eb249bbf'}]

In [24]:
import pandas as pd

In [25]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I found out abo...,74eb249bbf
1,"If I join now, am I still eligible for a certi...",74eb249bbf
2,What do I need to do to get a certificate if I...,74eb249bbf
3,Is it okay to start the course after it alread...,74eb249bbf
4,Can late joiners still submit the project for ...,74eb249bbf


In [11]:
from evaluation_utils import llm_structured_retry

In [12]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [15]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [18]:
from evaluation_utils import calc_price,calc_total_price

In [19]:
calc_total_price(usages)

0.0035385000000000004

In [20]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [21]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [33]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07834125000000002

In [34]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07834125000000002

In [35]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [36]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [37]:
len(df_ground_truth)

515